# 🤪 Conditional WGAN-GP on CelebA Faces - Train

In this notebook, we'll walk through the steps required to train your own Conditional GAN on the CelebA faces dataset

The code has been adapted from the excellent [CGAN tutorial](https://keras.io/examples/generative/conditional_gan/) created by Sayak Paul, available on the Keras website.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from collections import defaultdict
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from safetensors.torch import load_file
from transformers import Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display

# The Hugging Face Trainer picks the best available device
# automatically, so we only report what it will find
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 64
CHANNELS = 3
CLASSES = 2
BATCH_SIZE = 128
Z_DIM = 32
LEARNING_RATE = 0.00005
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.999
EPOCHS = 20
CRITIC_STEPS = 3
GP_WEIGHT = 10.0
LOAD_MODEL = False
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.9
LABEL = "Blond_Hair"

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the label dataset
attributes = pd.read_csv("../../../data/celeba-dataset/list_attr_celeba.csv")
print(attributes.columns)
attributes.head()

In [ ]:
# Load the data
labels = attributes[LABEL].tolist()
int_labels = [x if x == 1 else 0 for x in labels]

In [ ]:
class ImageDataset(Dataset):
    """
    All image files found under `folder` (searched recursively, in sorted order),
    resized to `image_size` - the PyTorch counterpart of Keras'
    image_dataset_from_directory. Images are uint8 tensors of shape (C, H, W),
    paired with labels[idx] when a list of labels (one per file) is given.
    """

    def __init__(self, folder, image_size, color_mode="rgb", labels=None):
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
        self.paths = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in exts)
        self.mode = {"grayscale": "L", "rgb": "RGB"}[color_mode]
        self.resize = transforms.Resize(
            image_size, interpolation=transforms.InterpolationMode.BILINEAR
        )
        self.labels = labels
        self.transform = None  # optional function applied to each image
        self.target_transform = None  # optional function applied to each label
        print(f"Found {len(self.paths)} files.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert(self.mode)
        img = transforms.functional.pil_to_tensor(self.resize(img))
        if self.transform:
            img = self.transform(img)
        # The Trainer passes each example to the model as keyword arguments,
        # so we return a dict whose keys match the model's forward()
        if self.labels is None:
            return {"pixel_values": img}
        label = self.labels[idx]
        if self.target_transform:
            label = self.target_transform(label)
        return {"pixel_values": img, "labels": label}


train_data = ImageDataset(
    "../../../data/celeba-dataset/img_align_celeba",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    color_mode="rgb",
    labels=int_labels,
)

In [ ]:
# Preprocess the data
def preprocess(img):
    """
    Normalize the images to [-1, 1]
    """
    img = (img.float() - 127.5) / 127.5
    return img


def one_hot(label):
    return F.one_hot(torch.tensor(label), num_classes=CLASSES).float()


# Like Keras' .map(): applied to every (image, label) pair as it is loaded.
# No DataLoader needed: the Trainer batches and shuffles the dataset itself.
train_data.transform = preprocess
train_data.target_transform = one_hot
train = train_data

In [ ]:
# Show some faces from the training set
# A DataLoader just to look at one batch (the Trainer makes its own for training)
train_sample = next(iter(DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)))["pixel_values"]

In [ ]:
display(train_sample, cmap=None)

## 2. Build the GAN <a name="build"></a>

In [ ]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        # PyTorch images are channels-first: (batch, channels, height, width).
        # padding=1 with kernel 4 / stride 2 halves the size, like Keras' padding="same".
        self.model = nn.Sequential(
            nn.Conv2d(CHANNELS + CLASSES, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.3),  # Keras' LeakyReLU() default slope is 0.3 (PyTorch's is 0.01)
            nn.Dropout(0.3),
            nn.Conv2d(128, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv2d(128, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv2d(128, 1, kernel_size=4, stride=1, padding=0),
            nn.Flatten(),
        )

    def forward(self, images, image_one_hot_labels):
        # Stack the label maps onto the colour channels: (batch, CHANNELS + CLASSES, H, W)
        x = torch.cat([images, image_one_hot_labels], dim=1)
        return self.model(x)


critic = Critic()
print(critic)
print(f"Total params: {sum(p.numel() for p in critic.parameters()):,}")

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Unflatten(1, (Z_DIM + CLASSES, 1, 1)),  # (batch, n) -> (batch, n, 1, 1)
            nn.ConvTranspose2d(Z_DIM + CLASSES, 128, kernel_size=4, stride=1, padding=0, bias=False),
            # PyTorch momentum = 1 - Keras momentum (Keras 0.9 -> PyTorch 0.1)
            nn.BatchNorm2d(128, momentum=0.1),
            nn.LeakyReLU(0.2),
            # padding=1 with kernel 4 / stride 2 doubles the size, like Keras' padding="same"
            nn.ConvTranspose2d(128, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128, momentum=0.1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128, momentum=0.1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64, momentum=0.1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, CHANNELS, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, z, one_hot_labels):
        # Append the one-hot label to the latent vector: (batch, Z_DIM + CLASSES)
        x = torch.cat([z, one_hot_labels], dim=1)
        return self.model(x)


generator = Generator()
print(generator)
print(f"Total params: {sum(p.numel() for p in generator.parameters()):,}")

In [ ]:
class ConditionalWGAN(nn.Module):
    def __init__(self, critic, generator, latent_dim, critic_steps, gp_weight):
        super().__init__()
        self.critic = critic
        self.generator = generator
        self.latent_dim = latent_dim
        self.critic_steps = critic_steps
        self.gp_weight = gp_weight

    def gradient_penalty(self, real_images, fake_images, image_one_hot_labels):
        batch_size = real_images.shape[0]
        # (the book samples alpha from a normal distribution;
        # the WGAN-GP paper uses a uniform one on [0, 1])
        alpha = torch.randn(batch_size, 1, 1, 1, device=real_images.device)
        diff = fake_images - real_images
        interpolated = (real_images + alpha * diff).requires_grad_(True)

        pred = self.critic(interpolated, image_one_hot_labels)

        # Gradient of the critic's output w.r.t. its input images.
        # create_graph=True lets us backpropagate through the penalty itself.
        grads = torch.autograd.grad(pred.sum(), interpolated, create_graph=True)[0]
        norm = torch.sqrt(torch.sum(grads**2, dim=[1, 2, 3]))
        gp = torch.mean((norm - 1.0) ** 2)
        return gp

    def critic_loss(self, pixel_values, labels):
        # labels are one-hot; repeat each over the image area: (batch, CLASSES, H, W)
        image_one_hot_labels = labels[:, :, None, None].expand(-1, -1, IMAGE_SIZE, IMAGE_SIZE)

        batch_size = pixel_values.shape[0]
        random_latent_vectors = torch.randn(batch_size, self.latent_dim, device=pixel_values.device)
        with torch.no_grad():  # the generator is not trained in this step
            fake_images = self.generator(random_latent_vectors, labels)

        fake_predictions = self.critic(fake_images, image_one_hot_labels)
        real_predictions = self.critic(pixel_values, image_one_hot_labels)

        c_wass_loss = fake_predictions.mean() - real_predictions.mean()
        c_gp = self.gradient_penalty(pixel_values, fake_images, image_one_hot_labels)
        c_loss = c_wass_loss + c_gp * self.gp_weight
        metrics = {"c_loss": c_loss.item(), "c_wass_loss": c_wass_loss.item(), "c_gp": c_gp.item()}
        return c_loss, metrics

    def forward(self, pixel_values, labels):
        # The generator loss - the "loss" the Trainer optimizes
        image_one_hot_labels = labels[:, :, None, None].expand(-1, -1, IMAGE_SIZE, IMAGE_SIZE)
        batch_size = pixel_values.shape[0]
        random_latent_vectors = torch.randn(batch_size, self.latent_dim, device=pixel_values.device)
        fake_images = self.generator(random_latent_vectors, labels)
        fake_predictions = self.critic(fake_images, image_one_hot_labels)
        g_loss = -fake_predictions.mean()
        return {"loss": g_loss}


class GANTrainer(Trainer):
    """
    The standard Trainer does one loss / one optimizer per batch. A conditional WGAN-GP trains
    two networks, each with its own optimizer, and updates the critic several times
    per generator update, so we override:
      - create_optimizer: one Adam for each network
      - training_step: CRITIC_STEPS critic updates, then one generator update
      - log: add our metrics to the Trainer's log
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.gan_metrics = defaultdict(list)

    def create_optimizer(self, model=None):
        lr, betas = self.args.learning_rate, (self.args.adam_beta1, self.args.adam_beta2)
        # self.optimizer is the one the Trainer steps for us: the generator's
        self.optimizer = torch.optim.Adam(self.model.generator.parameters(), lr=lr, betas=betas)
        # The critic's optimizer we step ourselves in training_step
        self.c_optimizer = torch.optim.Adam(self.model.critic.parameters(), lr=lr, betas=betas)
        return self.optimizer

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)  # move the batch to the device

        # Train the critic several times per generator update
        for i in range(model.critic_steps):
            c_loss, metrics = model.critic_loss(**inputs)
            self.c_optimizer.zero_grad()
            self.accelerator.backward(c_loss)
            self.c_optimizer.step()

        # Train the generator: we only compute the gradients here; the Trainer
        # then calls self.optimizer.step() and zero_grad(), as for any model
        outputs = model(**inputs)
        self.accelerator.backward(outputs["loss"])

        metrics["g_loss"] = outputs["loss"].item()
        for name, value in metrics.items():
            self.gan_metrics[name].append(value)
        return outputs["loss"].detach()

    def log(self, logs, *args, **kwargs):
        # Replace the Trainer's "loss" (which is just g_loss) by our averaged metrics
        if self.gan_metrics and "loss" in logs:
            logs.pop("loss")
            logs.update({name: float(np.mean(v)) for name, v in self.gan_metrics.items()})
            self.gan_metrics.clear()
        super().log(logs, *args, **kwargs)

In [ ]:
# Create a GAN
cgan = ConditionalWGAN(
    critic=critic,
    generator=generator,
    latent_dim=Z_DIM,
    critic_steps=CRITIC_STEPS,
    gp_weight=GP_WEIGHT,
)

In [ ]:
if LOAD_MODEL:
    # The Trainer saves each checkpoint as ./checkpoint/checkpoint-<step>/model.safetensors
    checkpoint_dir = get_last_checkpoint("./checkpoint")
    cgan.load_state_dict(load_file(os.path.join(checkpoint_dir, "model.safetensors")))

## 3. Train the GAN <a name="train"></a>

In [ ]:
# Keras' steps_per_epoch=1: an "epoch" is just 1 batch. The Trainer has no such
# option, so we train for EPOCHS * 100 steps and log / save / make images every step.
steps_per_epoch = 1

# TrainingArguments holds the optimizer and training settings
# (GANTrainer.create_optimizer builds one Adam per network from them)
training_args = TrainingArguments(
    output_dir="./checkpoint",
    learning_rate=LEARNING_RATE,
    adam_beta1=ADAM_BETA_1,
    adam_beta2=ADAM_BETA_2,
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    max_grad_norm=0.0,  # the Trainer clips gradients by default; Keras doesn't
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=EPOCHS * 100 * steps_per_epoch,
    logging_strategy="steps",
    logging_steps=steps_per_epoch,
    save_strategy="steps",
    save_steps=steps_per_epoch,
    save_only_model=True,  # weights only, like save_weights_only=True
    save_total_limit=1,  # keep just the latest checkpoint
    report_to="tensorboard",
    remove_unused_columns=False,  # our dataset keys are already what the model needs
)

In [ ]:
# TensorBoard logs go to ./logs (set with an environment variable in transformers 5)
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"


class ImageGenerator(TrainerCallback):
    def __init__(self, num_img, latent_dim):
        self.num_img = num_img
        self.latent_dim = latent_dim

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % steps_per_epoch != 0:  # only at the end of an "epoch"
            return
        epoch = state.global_step // steps_per_epoch - 1
        random_latent_vectors = torch.randn(self.num_img, self.latent_dim, device=args.device)
        generator = model.generator
        generator.eval()  # BatchNorm uses its running statistics, like Keras' training=False
        for label, one_hot_label in [(0, [1.0, 0.0]), (1, [0.0, 1.0])]:
            class_label = torch.tensor([one_hot_label], device=args.device).repeat(self.num_img, 1)
            with torch.no_grad():
                generated_images = generator(random_latent_vectors, class_label)
            generated_images = generated_images * 127.5 + 127.5
            display(
                generated_images,
                save_to="./output/generated_img_%03d_label_%d.png" % (epoch, label),
                cmap=None,
            )
        generator.train()

In [ ]:
trainer = GANTrainer(
    model=cgan,
    args=training_args,
    train_dataset=train,
    callbacks=[ImageGenerator(num_img=10, latent_dim=Z_DIM)],
)
history = trainer.train()

In [ ]:
# Save the final models
torch.save(generator.state_dict(), "./models/generator.pt")
torch.save(critic.state_dict(), "./models/critic.pt")

## Generate images

In [ ]:
# 0 label
z_sample = np.random.normal(size=(10, Z_DIM))
class_label = np.repeat([[1, 0]], 10, axis=0)
cgan.generator.eval()  # BatchNorm in inference mode, like Keras' predict()
with torch.no_grad():
    imgs = cgan.generator(
        torch.tensor(z_sample, dtype=torch.float32, device=trainer.args.device),
        torch.tensor(class_label, dtype=torch.float32, device=trainer.args.device),
    )
display(imgs, cmap=None)

In [ ]:
# 1 label
z_sample = np.random.normal(size=(10, Z_DIM))
class_label = np.repeat([[0, 1]], 10, axis=0)
cgan.generator.eval()
with torch.no_grad():
    imgs = cgan.generator(
        torch.tensor(z_sample, dtype=torch.float32, device=trainer.args.device),
        torch.tensor(class_label, dtype=torch.float32, device=trainer.args.device),
    )
display(imgs, cmap=None)